# Gatefall — LoRA training on Google Colab (kohya-ss/sd-scripts, free GPU)

Trains a per-character LoRA against Pony Diffusion V6 XL (or
Illustrious) so future generations of that character stay consistent
across pose/crop/expression, instead of drifting the way a bare
fixed-seed prompt does once composition tokens change.

**Before running:** `Runtime` -> `Change runtime type` -> `T4 GPU` ->
`Save`. Run cells in order.

**Prerequisite — you need a training set first**, not just the single
locked model-sheet image. See `docs/art-direction.md` and the
img2img workflow discussed there: generate 15-20 variants of the
locked character (different angles, expressions, crops) via img2img
at low denoise off the locked reference image, so the outfit/face
stay close across the set. Fresh txt2img rerolls, even at the same
seed, are not consistent enough once the prompt's composition tokens
change — that's the exact problem this LoRA fixes going forward.

**Colab free-tier limits apply** — sessions disconnect after
inactivity and there's a rolling GPU-time cap. LoRA training (unlike
a single image generation) can take 20-60+ minutes depending on
dataset size and epoch count, so budget for that.

## 1. Confirm the GPU is attached

In [1]:
!nvidia-smi

Sat Sep  5 15:21:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
# Create a clean Python 3.11 virtual environment
!pip install -q uv
!uv python install 3.11
!uv venv /content/sdvenv --python 3.11 --seed --clear

Python 3.11 is already installed
Using CPython 3.11.16
Creating virtual environment with seed packages at: /content/sdvenv
 + packaging==26.3
 + pip==26.2.1
 + setuptools==84.0.0
 + wheel==0.48.0
Activate with: source /content/sdvenv/bin/activate


In [9]:
!uv pip install \
  --python /content/sdvenv/bin/python \
  torch==2.6.0 torchvision==0.21.0 \
  --index-url https://download.pytorch.org/whl/cu124

Using Python 3.11.16 environment at: /content/sdvenv
Resolved 26 packages in 237ms
Installed 26 packages in 307ms
 + filelock==3.32.3
 + fsspec==2026.7.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + numpy==2.4.6
 + nvidia-cublas-cu12==12.4.5.8
 + nvidia-cuda-cupti-cu12==12.4.127
 + nvidia-cuda-nvrtc-cu12==12.4.127
 + nvidia-cuda-runtime-cu12==12.4.127
 + nvidia-cudnn-cu12==9.1.0.70
 + nvidia-cufft-cu12==11.2.1.3
 + nvidia-curand-cu12==10.3.5.147
 + nvidia-cusolver-cu12==11.6.1.9
 + nvidia-cusparse-cu12==12.3.1.170
 + nvidia-cusparselt-cu12==0.6.2
 + nvidia-nccl-cu12==2.21.5
 + nvidia-nvjitlink-cu12==12.4.127
 + nvidia-nvtx-cu12==12.4.127
 + pillow==12.3.0
 + sympy==1.13.1
 + torch==2.6.0+cu124
 + torchvision==0.21.0+cu124
 + triton==3.2.0
 + typing-extensions==4.16.0


In [26]:
import os

%cd /content/sd-scripts

# 1. Install general requirements and ensure scipy/accelerate are present
!uv pip install \
  --python /content/sdvenv/bin/python \
  --upgrade --force-reinstall \
  -r requirements.txt accelerate scipy

# 2. Reinstall the specific PyTorch stack for CUDA 13.0 to fix the torchvision::nms error
# We use the cu130 index to get the binaries that include the necessary C++ operators
!uv pip install \
  --python /content/sdvenv/bin/python \
  --upgrade --force-reinstall \
  torch torchvision torchaudio \
  --index-url https://download.pytorch.org/whl/cu130

/content/sd-scripts
Using Python 3.11.16 environment at: /content/sdvenv
Resolved 77 packages in 793ms
Prepared 77 packages in 222ms
Uninstalled 77 packages in 359ms
Installed 77 packages in 310ms
 ~ absl-py==2.5.0
 ~ accelerate==1.6.0
 ~ bitsandbytes==0.50.2
 ~ certifi==2026.7.22
 ~ charset-normalizer==3.5.1
 ~ cuda-bindings==13.3.1
 - cuda-pathfinder==1.6.0
 + cuda-pathfinder==1.8.1
 ~ cuda-toolkit==13.0.3.0
 ~ diffusers==0.32.1
 ~ einops==0.7.0
 - filelock==3.32.3
 + filelock==3.32.5
 ~ fsspec==2026.7.0
 ~ ftfy==6.3.1
 ~ grpcio==1.83.1
 ~ hf-xet==1.6.0
 ~ huggingface-hub==0.34.3
 ~ idna==3.19
 ~ imagesize==1.4.1
 ~ importlib-metadata==9.0.1
 ~ jinja2==3.1.6
 ~ library==0.0.0 (from file:///content/sd-scripts)
 ~ lion-pytorch==0.2.3
 ~ markdown==3.10.3
 ~ markdown-it-py==4.2.0
 ~ markupsafe==3.0.3
 ~ mdurl==0.1.2
 ~ mpmath==1.3.0
 ~ networkx==3.6.1
 ~ numpy==2.4.6
 ~ nvidia-cublas==13.1.1.3
 ~ nvidia-cuda-cupti==13.0.85
 ~ nvidia-cuda-nvrtc==13.0.88
 ~ nvidia-cuda-runtime==13.0.96
 ~ 

In [27]:
import sys
# Ensure the virtual environment site-packages are first in the path
venv_path = '/content/sdvenv/lib/python3.11/site-packages'
if venv_path not in sys.path:
    sys.path.insert(0, venv_path)

!/content/sdvenv/bin/python --version
!/content/sdvenv/bin/python -c "import torch; print('PyTorch:', torch.__version__); print('CUDA available:', torch.cuda.is_available())"
!/content/sdvenv/bin/python -c "import numpy; print('NumPy:', numpy.__version__)"
!/content/sdvenv/bin/python -c "import scipy; print('SciPy:', scipy.__version__)"
!/content/sdvenv/bin/python -c "import torchvision; print('Torchvision:', torchvision.__version__)"

Python 3.11.16
PyTorch: 2.14.0+cu130
CUDA available: True
NumPy: 2.4.6
SciPy: 1.17.1
Torchvision: 0.29.0+cu130


In [28]:
import os
import subprocess

# Validation check using the venv python directly
venv_python = "/content/sdvenv/bin/python"
check_cmd = [venv_python, "-c", "import torchvision; from library import config_util; print('Success: sd-scripts library imported with torchvision ops.')"]

try:
    os.chdir("/content/sd-scripts")
    result = subprocess.run(check_cmd, capture_output=True, text=True, check=True)
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print("Validation failed:")
    print(e.stderr)
    if "torchvision::nms" in e.stderr:
        print("\nTIP: The nms operator error sometimes requires a 'Restart Session' in Colab to clear the dynamic linker cache, even after a correct reinstall.")

Success: sd-scripts library imported with torchvision ops.



## 2. Install kohya-ss/sd-scripts

In [2]:
!pip install -U pip setuptools wheel
!pip install "numpy>=1.26,<2" "scipy==1.11.4"

  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 82.0 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> scipy

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [3]:
%cd /content
!git clone https://github.com/kohya-ss/sd-scripts
%cd /content/sd-scripts
!pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124 -q
!pip install --upgrade -r requirements.txt -q
!pip install accelerate -q

/content
Cloning into 'sd-scripts'...
remote: Enumerating objects: 11693, done.
remote: Counting objects: 100% (213/213), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 11693 (delta 155), reused 121 (delta 121), pack-reused 11480 (from 2)
Receiving objects: 100% (11693/11693), 30.37 MiB | 19.52 MiB/s, done.
Resolving deltas: 100% (8329/8329), done.
/content/sd-scripts
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for library (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.20.0 requires tens

## 3. Mount Google Drive

Used for three things: reading the base checkpoint (reuse the one you
already have in Drive from the ComfyUI notebook, if you put it there),
reading your training image set, and saving the finished LoRA
somewhere that survives the session ending.

In [29]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 4. Point to the base checkpoint

Train against the **same checkpoint** you generated the training
images with (Pony Diffusion V6 XL, per `docs/art-direction.md`) —
training against a mismatched base checkpoint gives worse results.

In [ ]:
CHECKPOINT_PATH = "/content/drive/MyDrive/gatefall-checkpoints/ponyDiffusionV6XL.safetensors"  # @param {type:"string"}

import os
assert os.path.exists(CHECKPOINT_PATH), f"Checkpoint not found at {CHECKPOINT_PATH} — fix the path (see the ComfyUI notebook's 3a/3b for how you got this file into Drive)."
print("Checkpoint found:", CHECKPOINT_PATH)

## 5. Set up the training image set

1. In Drive, create a folder for this character's training images,
   e.g. `gatefall-lora-training/faelen/`.
2. Put your 15-20 img2img-generated images directly in that folder.
3. For **each** image, add a matching `.txt` caption file with the
   same base filename (e.g. `faelen_01.png` needs `faelen_01.txt`).
   Caption format: a short **trigger word** unique to this character
   (something the base model won't already associate with anything —
   e.g. `flnwarden`), followed by tags for what's *different* in that
   specific image (pose, expression, crop, background) — leave out
   the constant identity traits (hair color, armor, elf ears) that
   are true in every image; the LoRA learns those from the trigger
   word plus the images themselves, not from restating them per file.
   Example `faelen_03.txt`:
   ```
   flnwarden, portrait, upper body, happy expression, slight smile
   ```
4. Update `TRAIN_DATA_DIR` below to match.

In [ ]:
TRAIN_DATA_DIR = "/content/drive/MyDrive/gatefall-lora-training/faelen"  # @param {type:"string"}
TRIGGER_WORD = "flnwarden"  # @param {type:"string"}

import os
images = [f for f in os.listdir(TRAIN_DATA_DIR) if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))]
captions = [f for f in os.listdir(TRAIN_DATA_DIR) if f.lower().endswith(".txt")]
print(f"Found {len(images)} images and {len(captions)} caption files in {TRAIN_DATA_DIR}")
missing = [f for f in images if os.path.splitext(f)[0] + ".txt" not in captions]
if missing:
    print("WARNING — these images have no matching .txt caption file:")
    for m in missing:
        print(" ", m)
else:
    print("Every image has a matching caption file. Good to proceed.")

Fix any warnings above before continuing — a missing caption file
silently gets skipped or errors out depending on trainer settings, so
it's worth catching now.

## 6. Write the dataset config

In [ ]:
NUM_REPEATS = 10  # @param {type:"integer"}

dataset_toml = f"""
[general]
caption_extension = '.txt'
shuffle_caption = true

[[datasets]]
resolution = 1024
batch_size = 1

  [[datasets.subsets]]
  image_dir = '{TRAIN_DATA_DIR}'
  num_repeats = {NUM_REPEATS}
"""

with open("/content/dataset_config.toml", "w") as f:
    f.write(dataset_toml)

print(dataset_toml)

## 7. Train

`network_train_unet_only` is on because it's recommended for SDXL
LoRA training (per sd-scripts' own SDXL docs). `sdpa` uses PyTorch's
built-in attention so no separate xformers install is needed. This is
the slow cell — 20-60+ minutes depending on dataset size and
`MAX_TRAIN_EPOCHS`; the Colab tab can be left in the background but
don't close it or let the session idle-disconnect.

In [ ]:
OUTPUT_NAME = "faelen_lora"  # @param {type:"string"}
OUTPUT_DIR = "/content/lora_output"  # @param {type:"string"}
MAX_TRAIN_EPOCHS = 10  # @param {type:"integer"}

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

%cd /content/sd-scripts
!accelerate launch --num_cpu_threads_per_process 1 sdxl_train_network.py \
  --pretrained_model_name_or_path="{CHECKPOINT_PATH}" \
  --dataset_config="/content/dataset_config.toml" \
  --output_dir="{OUTPUT_DIR}" \
  --output_name="{OUTPUT_NAME}" \
  --save_model_as=safetensors \
  --network_module=networks.lora \
  --network_dim=32 \
  --network_alpha=16 \
  --network_train_unet_only \
  --learning_rate=1e-4 \
  --optimizer_type="AdamW8bit" \
  --lr_scheduler="cosine" \
  --max_train_epochs={MAX_TRAIN_EPOCHS} \
  --save_every_n_epochs=2 \
  --mixed_precision="fp16" \
  --gradient_checkpointing \
  --cache_text_encoder_outputs \
  --no_half_vae \
  --sdpa

## 8. Save the trained LoRA to Drive (so it survives the session)

In [ ]:
import shutil, glob, os

DRIVE_LORA_DIR = "/content/drive/MyDrive/gatefall-loras"  # @param {type:"string"}
os.makedirs(DRIVE_LORA_DIR, exist_ok=True)

for f in glob.glob(os.path.join(OUTPUT_DIR, "*.safetensors")):
    shutil.copy(f, DRIVE_LORA_DIR)
    print("Saved:", os.path.join(DRIVE_LORA_DIR, os.path.basename(f)))

## 9. Use it in ComfyUI

1. Get the `.safetensors` file from `gatefall-loras/` in Drive into
   `ComfyUI/models/loras/` — in the ComfyUI Colab notebook, mount
   Drive the same way (3b's pattern) and symlink/copy it in, or
   `wget`/copy it directly since it's already in your own Drive.
2. In the ComfyUI graph, add a **`LoraLoader`** node between
   `Load Checkpoint` and the rest of the graph (it takes the
   checkpoint's MODEL/CLIP outputs as input and passes modified
   versions onward — rewire `KSampler` and both `CLIP Text Encode`
   nodes to pull from `LoraLoader`'s outputs instead of
   `Load Checkpoint`'s directly).
3. Select the trained LoRA file in that node, set strength around
   `0.7-0.9` to start.
4. Include the trigger word (`flnwarden` if you kept the default
   above) in your prompt — that's what activates the learned
   character identity.
5. Now pose/crop/expression prompts should hold the design far more
   reliably than the bare fixed-seed approach did.